# Trening bazowego U-Neta

Czteropoziomowy U-Net rozwija model z notebooków 05–06. W tym etapie porównaliśmy dwa learning rate'y, losowe odbicie poziome i trzy ziarna treningowe dla wariantu bez augmentacji. Zapisane wyniki służą jako punkt odniesienia dla dalszych eksperymentów na LGG MRI.

Domyślne uruchomienie pokazuje podsumowanie zapisanych przebiegów. Pełne historie są w `outputs/`, a najlepsze checkpointy w `outputs/checkpoints/`.

## 1. Biblioteki, ścieżki i konfiguracja

Ścieżki wyznaczamy względem katalogu projektu. Aby rozpocząć nowy trening, ustaw `run_training = True` i nadaj mu własne `run_name`. Pętla sprawdza obie ścieżki wyjściowe przed inicjalizacją modelu.

In [1]:
from pathlib import Path
import sys

import torch
import pandas as pd
import tifffile
from torch.utils.data import Dataset, DataLoader

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import UNet

In [2]:
run_training = False
run_name = "unet_bce_dice_lr3e-4_seed44_repeat"
seed = 44
learning_rate = 3e-4
batch_size = 4
num_epochs = 40
use_horizontal_flip = False

## 2. Dane i preprocessing

Łączymy metadane z zapisanym podziałem pacjentów. Train zawiera 2714 przekrojów, a val 615. `MRIDataset` wczytuje TIFF-y na żądanie i zwraca obraz `(3, 256, 256)` w zakresie `[0, 1]` oraz maskę `(1, 256, 256)` z wartościami 0 lub 1.

W przebiegu z odbiciem jedno losowanie decyduje o przekształceniu obrazu i maski razem. Walidacja używa oryginalnych przekrojów.

In [3]:
pairs_df = pd.read_csv(project_root / "data" / "metadata" / "pairs.csv")
splits_df = pd.read_csv(project_root / "data" / "splits" / "splits.csv")
pairs_with_split = pairs_df.merge(
    splits_df, on="patient_id", how="left", validate="many_to_one"
)

val_df = pairs_with_split.loc[pairs_with_split["split"] == "val"].copy()

train_df = pairs_with_split.loc[pairs_with_split["split"] == "train"].copy()
print(f"Train: {len(train_df)} przekrojów, val: {len(val_df)} przekrojów.")

Train: 2714 przekrojów, val: 615 przekrojów.


In [4]:
def preprocess_pair(image, mask):
    image_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255
    mask_tensor = torch.from_numpy(mask > 0).float().unsqueeze(0)

    return (image_tensor, mask_tensor)

class MRIDataset(Dataset):
    def __init__(self, pairs_df, project_root, transform=None):
        self.pairs_df = pairs_df
        self.project_root = project_root
        self.transform = transform

    def __len__(self):
        return self.pairs_df.shape[0]

    def __getitem__(self, index):
        row = self.pairs_df.iloc[index]
        image_path = self.project_root / row["image_path"]
        mask_path = self.project_root / row["mask_path"]
        image, mask = preprocess_pair(tifffile.imread(image_path), tifffile.imread(mask_path))

        if self.transform is not None:
            return self.transform(image, mask)
        return image, mask

In [5]:
def random_horizontal_flip(image, mask, p=0.5):
    if torch.rand(()) < p:
        image = torch.flip(image, dims=(2,))
        mask = torch.flip(mask, dims=(2,))

    return (image, mask)

## 3. Strata, trening i walidacja

BCE dostaje surowe logity. Soft Dice liczymy po sigmoidzie dla każdego przekroju i uśredniamy w paczce. Obie składowe straty mają wagę 1.

W treningu Adam aktualizuje wagi po każdej paczce. W walidacji progowanie `sigmoid(logits) > 0.5` daje maskę binarną. Dice liczymy po przekrojach, osobno dla całego zbioru, masek z adnotacją i masek pustych. Dwie puste maski dają Dice 1.

In [6]:
def soft_dice_loss(logits, targets, eps=1e-5):
    p = torch.sigmoid(logits)

    num = 2*(p*targets).sum(dim=(1, 2, 3)) + eps
    den = p.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + eps

    return 1 - (num/den).mean()

bce_criterion = torch.nn.BCEWithLogitsLoss()

def bce_dice_loss(logits, targets):
    bce = bce_criterion(logits, targets)
    soft_dice = soft_dice_loss(logits, targets)

    return bce + soft_dice

In [7]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    n_samples = 0

    for images, masks in loader:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, masks)

        loss.backward()
        optimizer.step()

        batch_size = images.shape[0]
        total_loss += loss.item() * batch_size
        n_samples += batch_size

    return total_loss / n_samples

In [8]:
def dice_per_sample(predictions, targets):
    targets_logic = targets > 0

    targets_sum = targets_logic.sum(dim=(1, 2, 3))
    predictions_sum = predictions.sum(dim=(1, 2, 3))
    intersection = (targets_logic & predictions).sum(dim=(1, 2, 3))

    denominator = targets_sum + predictions_sum

    return torch.where(
        denominator == 0,
        1.0,
        2 * intersection / denominator.clamp_min(1)
    )


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    n_samples = 0
    total_dice = 0.0
    total_dice_annotated = 0.0
    total_dice_empty = 0.0
    n_annotated = 0
    n_empty = 0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = criterion(logits, masks)

            predictions = torch.sigmoid(logits) > 0.5
            dice_scores = dice_per_sample(predictions, masks)
            has_annotation = masks.sum(dim=(1, 2, 3)) > 0

            batch_size = images.shape[0]
            total_loss += loss.item() * batch_size

            n_samples += batch_size
            n_annotated += has_annotation.sum().item()
            n_empty += (~has_annotation).sum().item()

            total_dice += dice_scores.sum().item()
            total_dice_annotated += dice_scores[has_annotation].sum().item()
            total_dice_empty += dice_scores[~has_annotation].sum().item()

    assert n_annotated + n_empty == n_samples

    return {
        "loss": total_loss / n_samples,
        "dice": total_dice / n_samples,
        "dice_annotated": total_dice_annotated / n_annotated,
        "dice_empty": total_dice_empty / n_empty
    }

## 4. Uruchomienie treningu

Każdy nowy przebieg tworzy świeży model, optymalizator i generator tasowania. Checkpoint zapisujemy, gdy rośnie walidacyjny Dice przekrojów z adnotacją. Pełną historię zapisujemy po ostatniej epoce.

In [9]:
if run_training:
    best_checkpoint_path = (
        project_root / "outputs" / "checkpoints" / f"{run_name}_best.pt"
    )
    history_path = project_root / "outputs" / f"{run_name}_history.csv"
    for path in (best_checkpoint_path, history_path):
        if path.exists():
            raise FileExistsError(f"Wybierz inną nazwę przebiegu. Plik już istnieje: {path}")
    best_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    train_generator = torch.Generator().manual_seed(seed)

    transform = random_horizontal_flip if use_horizontal_flip else None
    train_dataset = MRIDataset(train_df, project_root, transform=transform)
    val_dataset = MRIDataset(val_df, project_root)
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, generator=train_generator
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    history = []
    best_val_dice = -float("inf")

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, bce_dice_loss, optimizer, device)
        val_metrics = validate_one_epoch(model, val_loader, bce_dice_loss, device)
        history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

        current_dice = val_metrics["dice_annotated"]
        print(f"Epoka {epoch}/{num_epochs}, train loss = {train_loss:.4f}")
        print("Walidacja", val_metrics)

        if current_dice > best_val_dice:
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "history": history,
                    "seed": seed,
                },
                best_checkpoint_path,
            )
            best_val_dice = current_dice
            print(f"Zapisano najlepszy model z epoki {epoch}, Dice = {current_dice:.4f}")

    pd.DataFrame(history).to_csv(history_path, index=False)
    print(f"Zapisano {len(history)} epok w {history_path.name}")

## 5. Porównanie zapisanych przebiegów

Każdy wiersz pokazuje wyniki walidacyjne epoki z najwyższym Dice przekrojów z adnotacją. Wszystkie przebiegi miały 40 epok, a każda z wybranych epok mieściła się w pierwszych 15. Ostatnia kolumna pokazuje, jak często podczas treningu pokrycie oznaczonych obszarów spadało do zera.

In [10]:
saved_runs = [
    (42, 1e-3, "brak", "unet_bce_dice_history_lr1e-3.csv"),
    (42, 3e-4, "brak", "unet_bce_dice_history_lr3e-4.csv"),
    (42, 3e-4, "odbicie p=0,5", "unet_bce_dice_best_lr3e-4_hflip.csv"),
    (43, 3e-4, "brak", "unet_bce_dice_best_lr3e-4_seed43.csv"),
    (44, 3e-4, "brak", "unet_bce_dice_best_lr3e-4_seed44.csv"),
]
summary_rows = []
missing_histories = []

for run_seed, run_lr, augmentation, filename in saved_runs:
    path = project_root / "outputs" / filename
    if not path.is_file():
        missing_histories.append(filename)
        continue

    run_history = pd.read_csv(path)
    best = run_history.loc[run_history["dice_annotated"].idxmax()]
    summary_rows.append({
        "seed": run_seed,
        "lr": run_lr,
        "augmentacja": augmentation,
        "epoki": len(run_history),
        "najlepsza epoka": int(best["epoch"]),
        "Dice ogólny": best["dice"],
        "Dice oznaczonych": best["dice_annotated"],
        "Dice pustych": best["dice_empty"],
        "epoki z Dice oznaczonych = 0": int(run_history["dice_annotated"].eq(0).sum()),
    })

if missing_histories:
    print("Brak lokalnych historii:", ", ".join(missing_histories))

summary = pd.DataFrame(summary_rows)
summary.round(4)

,seed,lr,augmentacja,epoki,najlepsza epoka,Dice ogólny,Dice oznaczonych,Dice pustych,epoki z Dice oznaczonych = 0
0,42,0.0010,brak,40,2,0.4972,0.3380,0.5963,21
1,42,0.0003,brak,40,6,0.4799,0.4625,0.4908,0
2,42,0.0003,"odbicie p=0,5",40,4,0.5638,0.3675,0.6860,7
3,43,0.0003,brak,40,15,0.5221,0.4835,0.5462,6
4,44,0.0003,brak,40,7,0.5165,0.4688,0.5462,0


## Wnioski

Przy seedzie 42 obniżenie `lr` z `1e-3` do `3e-4` podniosło najlepszy Dice oznaczonych przekrojów z 0,3380 do 0,4625. W pojedynczym przebiegu odbicie poziome z `p=0.5` zwiększyło Dice pustych masek, ale obniżyło Dice oznaczonych.

Wariant bez augmentacji z `lr=3e-4` dał na trzech seedach średni najlepszy Dice oznaczonych 0,4716 ± 0,0108 (średnia ± odchylenie standardowe). To model bazowy dla dalszego strojenia. Notebook 08 pozwala ocenić jego checkpointy także po całych wolumenach pacjentów.